# OpenCL vs Metal benchmark (Apple Silicon)

Separates **kernel compute** from **H2D/D2H transfer** for `OCL_T` and `METAL_T`.

- **Copy**: bandwidth / launch overhead (`dst[i] = src[i]`)
- **CheckU**: multi-buffer + light math ([`K_CheckU`](../idpy/LBM/LBMKernels.py))
- **Transfer**: OCL buffer copy vs Metal `H2D(external)` memcpy; Metal `D2H` / zero-copy (`.host`) in the table only (near free)

**Timers:** On macOS, OpenCL `DeployProfiling` uses host wall clock (enqueue + wait), like Metal around `wait_until_completed` — Apple OpenCL event timestamps are unreliable. Elsewhere, OpenCL uses device event profiles. Both exclude compile (warmup after first build).

**Stats:** each point is mean ± SEM over `N_TRIALS` trial medians (each median from `N_INNER` timed launches after warmup).

**Density:** `t_site = t / V` (ns/site), same idea as Ising flip time = total time / (½ volume).


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../")

import numpy as np
import matplotlib.pyplot as plt

from idpy.core import OCL_T, METAL_T, IDPY_T, idpy_langs_sys, GetTenet
from idpy.core import IdpyMemory
from idpy.core.IdpyCode import IdpyKernel
from idpy.physics.lbm.LBMKernels import K_CheckU
from idpy.core.utils.CustomTypes import CustomTypes
from idpy.core.utils.SimpleTiming import SimpleTiming

In [ ]:
BLOCK = 128
N_WARM = 5
N_INNER = 8       # timed launches per trial → median
N_TRIALS = 16     # outer trials → mean ± SEM of medians
COPY_Vs = [2 ** 20, 2 ** 22, 2 ** 24, 2 ** 26]
CHECKU_Ls = [64, 128, 256, 512]
LANGS = [lang for lang in (OCL_T, METAL_T) if idpy_langs_sys.get(lang)]
print("Backends:", LANGS)
print(f"Stats: {N_TRIALS} trials × median of {N_INNER} (warmup {N_WARM})")

FP32 = CustomTypes({
    "PopType": "float", "NType": "float", "UType": "float",
    "SType": "int", "FlagType": "unsigned char",
})


def make_tenet(lang):
    return GetTenet({"lang": lang, "device": 0, "cl_kind": "gpu"})


def grid_block(V, block=BLOCK):
    grid = ((V + block - 1) // block, 1, 1)
    return grid, (block, 1, 1)


def mean_sem(xs):
    xs = np.asarray(xs, dtype=float)
    n = len(xs)
    mean = float(np.mean(xs))
    sem = float(np.std(xs, ddof=1) / np.sqrt(n)) if n > 1 else 0.0
    return mean, sem


def stable_profile(idea, args, n_warm=N_WARM, n_inner=N_INNER, n_trials=N_TRIALS):
    for _ in range(n_warm):
        idea.DeployProfiling(args)
    trial_medians = []
    for _ in range(n_trials):
        times = [idea.DeployProfiling(args)[1] for _ in range(n_inner)]
        trial_medians.append(float(np.median(times)))
    return mean_sem(trial_medians)


def stable_wall(fn, n_warm=N_WARM, n_inner=N_INNER, n_trials=N_TRIALS):
    for _ in range(n_warm):
        fn()
    trial_medians = []
    st = SimpleTiming()
    for _ in range(n_trials):
        times = []
        for _ in range(n_inner):
            st.Start()
            fn()
            st.End()
            times.append(st.GetElapsedTime()["time_s"])
        trial_medians.append(float(np.median(times)))
    return mean_sem(trial_medians)


def gb_s(nbytes, t_s):
    return (nbytes / t_s) / 1e9 if t_s > 0 else float("nan")


def gb_s_sem(nbytes, t_s, t_sem):
    if t_s <= 0:
        return float("nan")
    return gb_s(nbytes, t_s) * (t_sem / t_s)


def ns_per_site(t_s, V):
    return (t_s / V) * 1e9 if V > 0 else float("nan")


def ns_per_site_sem(t_sem, V):
    return (t_sem / V) * 1e9 if V > 0 else float("nan")


def fmt_pm(mean, sem, prec=4):
    return f"{mean:.{prec}g}±{sem:.{prec}g}"


def print_table(rows, headers):
    widths = [max(len(str(h)), max((len(f"{r[i]:.4g}" if isinstance(r[i], float) else str(r[i])) for r in rows), default=0))
              for i, h in enumerate(headers)]
    fmt = "  ".join(f"{{:{w}}}" for w in widths)
    print(fmt.format(*headers))
    print(fmt.format(*["-" * w for w in widths]))
    for r in rows:
        cells = []
        for x in r:
            cells.append(f"{x:.4g}" if isinstance(x, float) else str(x))
        print(fmt.format(*cells))


def plot_scaling(xs, series, xlabel, ylabel, title, xscale="log", yscale="log", yerr=None, dashed_series=None):
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for label, ys in series.items():
        err = yerr.get(label) if yerr else None
        ax.errorbar(xs, ys, yerr=err, marker="o", label=label, capsize=3)
    if dashed_series:
        for label, ys in dashed_series.items():
            ax.plot(xs, ys, linestyle="--", label=label)
    ax.set_xscale(xscale)
    ax.set_yscale(yscale)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()


## 1. Copy kernel (bandwidth / launch)

Bytes touched ≈ `2 * V * 4` (read src + write dst). Density: `t_site = t / V` (mean ± SEM).


In [ ]:
class K_Copy(IdpyKernel):
    def __init__(self, custom_types=None, constants=None, optimizer_flag=None):
        IdpyKernel.__init__(
            self,
            custom_types=custom_types or {},
            constants=constants or {},
            optimizer_flag=optimizer_flag,
        )
        self.SetCodeFlags("g_tid")
        self.params = {
            "PopType * dst": ["global", "restrict"],
            "PopType * src": ["global", "restrict", "const"],
        }
        self.kernels[IDPY_T] = """
        if(g_tid < V){
            dst[g_tid] = src[g_tid];
        }
        """


copy_results = {lang: [] for lang in LANGS}  # list of dicts per V

for lang in LANGS:
    tenet = make_tenet(lang)
    for V in COPY_Vs:
        grid, block = grid_block(V)
        src = IdpyMemory.Zeros(V, dtype=np.float32, tenet=tenet)
        dst = IdpyMemory.Zeros(V, dtype=np.float32, tenet=tenet)
        src.H2D(np.arange(V, dtype=np.float32) % 100)
        k = K_Copy(
            custom_types=FP32.Push(),
            constants={"V": V},
            optimizer_flag=True,
        )
        idea = k(tenet=tenet, grid=grid, block=block)
        t, t_sem = stable_profile(idea, [dst, src])
        nbytes = 2 * V * 4
        t_site = ns_per_site(t, V)
        t_site_sem = ns_per_site_sem(t_sem, V)
        gbs = gb_s(nbytes, t)
        gbs_sem = gb_s_sem(nbytes, t, t_sem)
        copy_results[lang].append({
            "V": V, "t": t, "t_sem": t_sem,
            "t_site_ns": t_site, "t_site_ns_sem": t_site_sem,
            "GB_s": gbs, "GB_s_sem": gbs_sem,
        })
        print(
            f"{lang} Copy V={V}: t={t*1e3:.3f}±{t_sem*1e3:.3f} ms  "
            f"{fmt_pm(t_site, t_site_sem)} ns/site  {fmt_pm(gbs, gbs_sem)} GB/s"
        )
    tenet.End()

rows = []
for i, V in enumerate(COPY_Vs):
    row = [V]
    for lang in LANGS:
        r = copy_results[lang][i]
        row += [
            fmt_pm(r["t"] * 1e3, r["t_sem"] * 1e3),
            fmt_pm(r["t_site_ns"], r["t_site_ns_sem"]),
            fmt_pm(r["GB_s"], r["GB_s_sem"]),
        ]
    rows.append(row)
headers = ["V"] + [f"{lang}:{h}" for lang in LANGS for h in ("ms", "ns/site", "GB/s")]
print()
print_table(rows, headers)

plot_scaling(
    COPY_Vs,
    {lang: [r["t_site_ns"] for r in copy_results[lang]] for lang in LANGS},
    xlabel="V (sites)",
    ylabel="ns / site",
    title="Copy: computational density",
    yerr={lang: [r["t_site_ns_sem"] for r in copy_results[lang]] for lang in LANGS},
)


## 2. CheckU (multi-buffer + light math)

`DIM=3`: reads/writes several `UType` fields per site. Density still `t / V` (mean ± SEM).


In [ ]:
DIM = 3
checku_results = {lang: [] for lang in LANGS}
checku_Vs = []

for lang in LANGS:
    tenet = make_tenet(lang)
    for L in CHECKU_Ls:
        V = L ** 3
        if lang == LANGS[0]:
            checku_Vs.append(V)
        grid, block = grid_block(V)
        u = IdpyMemory.Zeros(V * DIM, dtype=np.float32, tenet=tenet)
        old_u = IdpyMemory.Zeros(V * DIM, dtype=np.float32, tenet=tenet)
        delta_u = IdpyMemory.Zeros(V, dtype=np.float32, tenet=tenet)
        max_u = IdpyMemory.Zeros(V, dtype=np.float32, tenet=tenet)
        rng = np.random.default_rng(0)
        u.H2D(rng.standard_normal(V * DIM).astype(np.float32) * 1e-3)
        old_u.H2D(rng.standard_normal(V * DIM).astype(np.float32) * 1e-3)
        k = K_CheckU(
            custom_types=FP32.Push(),
            constants={"V": V, "DIM": DIM},
            optimizer_flag=True,
        )
        idea = k(tenet=tenet, grid=grid, block=block)
        args = [delta_u, old_u, max_u, u]
        t, t_sem = stable_profile(idea, args)
        # rough traffic: read u+old_u (2*DIM*V), write old_u+delta_u+max_u ((DIM+2)*V)
        nbytes = (2 * DIM + DIM + 2) * V * 4
        t_site = ns_per_site(t, V)
        t_site_sem = ns_per_site_sem(t_sem, V)
        gbs = gb_s(nbytes, t)
        gbs_sem = gb_s_sem(nbytes, t, t_sem)
        checku_results[lang].append({
            "L": L, "V": V, "t": t, "t_sem": t_sem,
            "t_site_ns": t_site, "t_site_ns_sem": t_site_sem,
            "GB_s": gbs, "GB_s_sem": gbs_sem,
        })
        print(
            f"{lang} CheckU L={L} V={V}: t={t*1e3:.3f}±{t_sem*1e3:.3f} ms  "
            f"{fmt_pm(t_site, t_site_sem)} ns/site  {fmt_pm(gbs, gbs_sem)} GB/s"
        )
    tenet.End()

rows = []
for i, L in enumerate(CHECKU_Ls):
    row = [L, checku_Vs[i]]
    for lang in LANGS:
        r = checku_results[lang][i]
        row += [
            fmt_pm(r["t"] * 1e3, r["t_sem"] * 1e3),
            fmt_pm(r["t_site_ns"], r["t_site_ns_sem"]),
            fmt_pm(r["GB_s"], r["GB_s_sem"]),
        ]
    rows.append(row)
headers = ["L", "V"] + [f"{lang}:{h}" for lang in LANGS for h in ("ms", "ns/site", "GB/s")]
print()
print_table(rows, headers)

plot_scaling(
    checku_Vs,
    {lang: [r["t_site_ns"] for r in checku_results[lang]] for lang in LANGS},
    xlabel="V = L^3 (sites)",
    ylabel="ns / site",
    title="CheckU: computational density",
    yerr={lang: [r["t_site_ns_sem"] for r in checku_results[lang]] for lang in LANGS},
)


## 3. Transfer (H2D / D2H)

Not discrete-GPU PCIe. OCL is a real buffer copy on Apple's OpenCL stack.

Metal:

- **`H2D(external)`** — `np.copyto` into persistent `.host` (memcpy; plotted as density).
- **`D2H()` / `H2D (zero-copy)`** — shared view / `.host` access; near free. Reported in the **table as absolute ms** only (not on the density plot — below `time.time()` resolution, so GB/s / ns/byte are not meaningful).

Density plot: `t_byte = t / nbytes` (ns/byte), mean ± SEM — OCL H2D/D2H and Metal H2D.


In [ ]:
xfer_results = {lang: {"H2D": [], "D2H": []} for lang in LANGS}
metal_zc = []  # Metal-only zero-copy (.host access); table only

for lang in LANGS:
    tenet = make_tenet(lang)
    for V in COPY_Vs:
        nbytes = V * 4
        host = np.arange(V, dtype=np.float32)
        dev = IdpyMemory.Zeros(V, dtype=np.float32, tenet=tenet)

        t_h2d, t_h2d_sem = stable_wall(lambda: dev.H2D(host))
        t_d2h, t_d2h_sem = stable_wall(lambda: dev.D2H())

        xfer_results[lang]["H2D"].append({
            "V": V, "nbytes": nbytes, "t": t_h2d, "t_sem": t_h2d_sem,
            "ns_byte": (t_h2d / nbytes) * 1e9,
            "ns_byte_sem": (t_h2d_sem / nbytes) * 1e9,
            "GB_s": gb_s(nbytes, t_h2d),
            "GB_s_sem": gb_s_sem(nbytes, t_h2d, t_h2d_sem),
        })
        xfer_results[lang]["D2H"].append({
            "V": V, "nbytes": nbytes, "t": t_d2h, "t_sem": t_d2h_sem,
            "ns_byte": (t_d2h / nbytes) * 1e9,
            "ns_byte_sem": (t_d2h_sem / nbytes) * 1e9,
            "GB_s": gb_s(nbytes, t_d2h),
            "GB_s_sem": gb_s_sem(nbytes, t_d2h, t_d2h_sem),
        })

        line = (
            f"{lang} V={V}: "
            f"H2D {t_h2d*1e3:.3f}±{t_h2d_sem*1e3:.3f} ms ({fmt_pm(gb_s(nbytes, t_h2d), gb_s_sem(nbytes, t_h2d, t_h2d_sem))} GB/s)  "
            f"D2H {t_d2h*1e3:.3f}±{t_d2h_sem*1e3:.3f} ms"
        )
        if lang != METAL_T:
            line += f" ({fmt_pm(gb_s(nbytes, t_d2h), gb_s_sem(nbytes, t_d2h, t_d2h_sem))} GB/s)"
        else:
            line += " (view; see table)"

        if lang == METAL_T:
            t_zc, t_zc_sem = stable_wall(lambda: dev.host)
            metal_zc.append({
                "V": V, "nbytes": nbytes, "t": t_zc, "t_sem": t_zc_sem,
            })
            line += f"  H2D(zc) {t_zc*1e3:.3f}±{t_zc_sem*1e3:.3f} ms"

        print(line)
    tenet.End()

rows = []
for i, V in enumerate(COPY_Vs):
    row = [V, V * 4]
    for lang in LANGS:
        h, d = xfer_results[lang]["H2D"][i], xfer_results[lang]["D2H"][i]
        row += [
            fmt_pm(h["t"] * 1e3, h["t_sem"] * 1e3),
            fmt_pm(h["GB_s"], h["GB_s_sem"]),
            fmt_pm(d["t"] * 1e3, d["t_sem"] * 1e3),
        ]
        if lang != METAL_T:
            row.append(fmt_pm(d["GB_s"], d["GB_s_sem"]))
        else:
            row.append("—")
    if metal_zc:
        z = metal_zc[i]
        row.append(fmt_pm(z["t"] * 1e3, z["t_sem"] * 1e3))
    rows.append(row)
headers = ["V", "bytes"]
for lang in LANGS:
    headers += [f"{lang}:H2D ms", f"{lang}:H2D GB/s", f"{lang}:D2H ms", f"{lang}:D2H GB/s"]
if metal_zc:
    headers += ["pymetallic:H2D(zc) ms"]
print()
print_table(rows, headers)

# Density plot: OCL H2D/D2H + Metal H2D only (Metal D2H / zc are table-only)
nbytes_axis = [V * 4 for V in COPY_Vs]
series = {}
yerr = {}
for lang in LANGS:
    series[f"{lang} H2D"] = [r["ns_byte"] for r in xfer_results[lang]["H2D"]]
    yerr[f"{lang} H2D"] = [r["ns_byte_sem"] for r in xfer_results[lang]["H2D"]]
    if lang != METAL_T:
        series[f"{lang} D2H"] = [r["ns_byte"] for r in xfer_results[lang]["D2H"]]
        yerr[f"{lang} D2H"] = [r["ns_byte_sem"] for r in xfer_results[lang]["D2H"]]
plot_scaling(
    nbytes_axis,
    series,
    xlabel="nbytes",
    ylabel="ns / byte",
    title="Transfer density (API cost)",
    yerr=yerr,
)


## How to read the plots

- **Falling `ns/site` with `V`**: launch / driver overhead amortized (small problems look “slow per site”).
- **Flat `ns/site`**: saturated regime — fair place to compare OCL vs Metal compute.
- **Error bars**: SEM of trial medians (compare backend gaps against them).
- **Transfer `ns/byte`**: OCL H2D/D2H and Metal H2D (memcpy). Metal `D2H` / zero-copy stay in the table as ms (too fast for density / GB/s). Not a PCIe story on M-series.
- Prefer large `V` when GPU History shows sustained activity.
